# 🧸 Experiment 02. Paired Label Transitions 🧸

This experiment examines how predictions for the same utterance change across conditions.

Using the preserved label files of the original study (n = 300), I pair each utterance's baseline label with its label under each replacement condition and classify every change against the human label.

## Contents

1. **Question**
2. **Data Preparation**
   - 2.1. Load and join the label files
   - 2.2. Verification against the scoring record
3. **Predictions**
4. **Paired Label Transitions**
   - 4.1. Replacement conditions
   - 4.2. Names-only condition (reference)
5. **Classification Errors**
   - 5.1. Error classification
   - 5.2. Most frequent transitions
6. **Observations**

## 1. Question

How do predictions for the same utterance change across conditions?

Experiment 01 compared the exclusion-rule replacement condition with baseline using summary statistics only.

| Measure | Baseline | Exclusion-rule replacement | Change |
|---|---:|---:|---:|
| Exact agreement count | 236 | 209 | −27 |
| Observed agreement ($P_o$) | 0.7867 | 0.6967 | −0.0900 |
| Cohen's κ | 0.6085 | 0.5023 | −0.1062 |

In the baseline confusion matrix, 20 utterances labeled NC by the human coder were labeled KET by the LLM. In the exclusion-rule replacement condition, this cell contains 40 utterances.

A confusion matrix compares each condition with the human labels separately. It does not show whether the same utterances changed, or whether a change made a prediction right or wrong.

The six conditions were run on the same 300 utterances, so the labels are paired. This experiment uses that pairing to trace, utterance by utterance, how each replacement condition moved labels away from baseline.

**Sources:** Experiment 01, Sections 1 and 3; [Main Scoring Record, Section 11](https://github.com/geekinthegit/coding-manual-ablation/blob/main/reports/main-scoring-record-2026-09-24.md) (exclusion-rule confusion matrix).

## 2. Data Preparation

This notebook reads two preserved files of the original study. Both are treated as read-only, and no new API calls are made.

| File | Content | Key |
|---|---|---|
| `data/scoring_labels.csv` | Human labels (`source_id`, `source_row_id`, `Tag`) | `source_id` |
| `runs/main-2026-09-24/final_labels.csv` | LLM final labels, long format (1,800 rows = 300 utterances × 6 conditions) | `utterance_id` |

The human `Tag` is converted to a category name with `TAG_TO_CATEGORY` from the original study's `scripts/tags.py`.

`scoring_labels.csv` is not tracked by git, so it is read from its original location rather than copied into this repository. The utterance text is not loaded. This notebook handles labels only.

#### Category Abbreviations

The following abbreviations are used throughout this repository.

| Abbreviation | Category |
|---|---|
| NC | Not coded |
| KET | Keeping Everyone Together |
| GSR | Getting Students to Relate |
| RST | Restating |
| REV | Revoicing |
| PFA | Pressing for Accuracy |
| PFR | Pressing for Reasoning |

### 2.1. Load and Join the Label Files

The code pivots `final_labels.csv` into one row per utterance with one column per condition, converts the human tags to abbreviated categories, and joins the two on the utterance ID.

The result is a 300-row table with the columns `utterance_id`, `human`, `baseline`, `definition_replacement`, `example_replacement`, `exclusion_rule_replacement`, `negative_control`, and `names_only`.

In [1]:
# 2.1. Load and join the label files

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Input paths (all read-only; nothing under ABLATION_ROOT is written)
ABLATION_ROOT = Path.home() / "coding-manual-ablation"
SCORING_LABELS = ABLATION_ROOT / "data" / "scoring_labels.csv"              # git-ignored: read in place, not copied
FINAL_LABELS = ABLATION_ROOT / "runs" / "main-2026-09-24" / "final_labels.csv"
SCRIPTS_DIR = ABLATION_ROOT / "scripts"

for path in (SCORING_LABELS, FINAL_LABELS, SCRIPTS_DIR):
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

# Reuse the original study's tag mapping and kappa implementation.
# Importing scorer also imports openai through its dependencies; no API key is read and no call is made.
sys.dont_write_bytecode = True        # do not write __pycache__ into the original repository
sys.path.insert(0, str(SCRIPTS_DIR))
from tags import TAG_TO_CATEGORY
from scorer import kappa

# Category abbreviations in tag order (NC, KET, GSR, RST, REV, PFA, PFR)
NAME_TO_ABBR = {
    "Not coded": "NC",
    "Keeping Everyone Together": "KET",
    "Getting Students to Relate": "GSR",
    "Restating": "RST",
    "Revoicing": "REV",
    "Pressing for Accuracy": "PFA",
    "Pressing for Reasoning": "PFR",
}
ABBR_TO_NAME = {abbr: name for name, abbr in NAME_TO_ABBR.items()}
CATEGORIES = [NAME_TO_ABBR[TAG_TO_CATEGORY[t]] for t in sorted(TAG_TO_CATEGORY)]
assert CATEGORIES == ["NC", "KET", "GSR", "RST", "REV", "PFA", "PFR"]

CONDITIONS = [
    "baseline",
    "definition_replacement",
    "example_replacement",
    "exclusion_rule_replacement",
    "negative_control",
    "names_only",
]
REPLACEMENTS = CONDITIONS[1:5]   # the four conditions compared with baseline in this notebook

# LLM labels: long format -> one row per utterance, one column per condition
final_long = pd.read_csv(FINAL_LABELS)
assert set(final_long["status"]) == {"resolved"}
model = (final_long
         .pivot(index="utterance_id", columns="condition", values="final_label")[CONDITIONS]
         .apply(lambda col: col.map(NAME_TO_ABBR)))
model.columns.name = None

# Human labels: Tag -> category name -> abbreviation, keyed by source_id
human_raw = pd.read_csv(SCORING_LABELS, keep_default_na=False, na_values=[""], usecols=["source_id", "Tag"])
assert human_raw["source_id"].is_unique
human = (human_raw.set_index("source_id")["Tag"].astype(int)
         .map(TAG_TO_CATEGORY).map(NAME_TO_ABBR).rename("human"))

# Join on source_id = utterance_id
labels = model.join(human, how="left").reset_index()
labels = labels[["utterance_id", "human", *CONDITIONS]]

# Verify the joined table
assert len(labels) == 300
assert labels.notna().all().all()
assert labels["utterance_id"].is_unique
assert set(labels["utterance_id"]) == set(final_long["utterance_id"])
assert set(labels["utterance_id"]) <= set(human.index)
for col in ["human", *CONDITIONS]:
    assert set(labels[col]) <= set(CATEGORIES), col

print(f"Rows: {len(labels)}, columns: {list(labels.columns)}")
print()
print(labels.head(5).to_string(index=False))

Rows: 300, columns: ['utterance_id', 'human', 'baseline', 'definition_replacement', 'example_replacement', 'exclusion_rule_replacement', 'negative_control', 'names_only']

 utterance_id human baseline definition_replacement example_replacement exclusion_rule_replacement negative_control names_only
          302    NC       NC                     NC                  NC                         NC               NC         NC
          848    NC       NC                     NC                  NC                         NC               NC         NC
         1024   KET      PFA                    PFA                 PFA                        PFA              RST        RST
         1079    NC       NC                     NC                  NC                         NC               NC        KET
         2294    NC       NC                     NC                  NC                         NC               NC        GSR


### 2.2. Verification Against the Scoring Record

Before analysing transitions, I reproduce the original per-condition statistics from the joined table using the original study's `kappa` function.

The values below are copied from the original scoring record. Exact agreement count, $P_o$, and $P_e$ come from Section 11; κ comes from Section 8.

| Condition | Exact agreement count | $P_o$ | $P_e$ | Cohen's κ |
|---|---:|---:|---:|---:|
| baseline | 236 | 0.7867 | 0.4551 | 0.6085 |
| definition_replacement | 235 | 0.7833 | 0.4492 | 0.6067 |
| example_replacement | 223 | 0.7433 | 0.4330 | 0.5473 |
| exclusion_rule_replacement | 209 | 0.6967 | 0.3906 | 0.5023 |
| negative_control | 230 | 0.7667 | 0.4374 | 0.5852 |
| names_only | 167 | 0.5567 | 0.3904 | 0.2727 |

The code checks that every computed value matches the record to four decimal places, and that the baseline confusion matrix equals the matrix typed by hand in Experiment 01.

**Source:** [Main Scoring Record, Sections 8 and 11](https://github.com/geekinthegit/coding-manual-ablation/blob/main/reports/main-scoring-record-2026-09-24.md)

In [2]:
# 2.2. Verification against the scoring record

# Values from the scoring record (Section 11: agree, P_o, P_e; Section 8: kappa)
record = pd.DataFrame({
    "agree": [236, 235, 223, 209, 230, 167],
    "P_o":   [0.7867, 0.7833, 0.7433, 0.6967, 0.7667, 0.5567],
    "P_e":   [0.4551, 0.4492, 0.4330, 0.3906, 0.4374, 0.3904],
    "kappa": [0.6085, 0.6067, 0.5473, 0.5023, 0.5852, 0.2727],
}, index=CONDITIONS)

# Baseline confusion matrix typed by hand in Experiment 01 (rows = human, columns = LLM)
matrix_01 = np.array([
    [171, 20, 1, 0, 6, 7, 2],
    [7, 22, 1, 1, 4, 4, 2],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 2, 0, 0, 0],
    [0, 0, 0, 1, 3, 0, 0],
    [1, 3, 0, 0, 0, 35, 3],
    [0, 0, 0, 0, 0, 0, 3],
])

human_names = labels["human"].map(ABBR_TO_NAME).tolist()
results = {}
for cond in CONDITIONS:
    r = kappa(human_names, labels[cond].map(ABBR_TO_NAME).tolist())
    assert r.n == 300
    assert r.agree == record.loc[cond, "agree"], cond
    assert round(r.p_o, 4) == record.loc[cond, "P_o"], cond
    assert round(r.p_e, 4) == record.loc[cond, "P_e"], cond
    assert round(r.kappa, 4) == record.loc[cond, "kappa"], cond
    results[cond] = r

# Baseline confusion matrix in tag order, compared with Experiment 01
name_order = [ABBR_TO_NAME[c] for c in CATEGORIES]
baseline_matrix = np.array([[results["baseline"].confusion[h][p] for p in name_order] for h in name_order])
assert np.array_equal(baseline_matrix, matrix_01)

print(f"{'Condition':<28} {'Agree':>5} {'Po':>8} {'Po rec':>8} {'Pe':>8} {'Pe rec':>8} {'Kappa':>8} {'K rec':>8}")
for cond in CONDITIONS:
    r, rec = results[cond], record.loc[cond]
    print(f"{cond:<28} {r.agree:>5} {r.p_o:>8.4f} {rec['P_o']:>8.4f} "
          f"{r.p_e:>8.4f} {rec['P_e']:>8.4f} {r.kappa:>8.4f} {rec['kappa']:>8.4f}")
print()
print("All six conditions match the scoring record to four decimal places.")
print("Baseline confusion matrix matches Experiment 01.")

Condition                    Agree       Po   Po rec       Pe   Pe rec    Kappa    K rec
baseline                       236   0.7867   0.7867   0.4551   0.4551   0.6085   0.6085
definition_replacement         235   0.7833   0.7833   0.4492   0.4492   0.6067   0.6067
example_replacement            223   0.7433   0.7433   0.4330   0.4330   0.5473   0.5473
exclusion_rule_replacement     209   0.6967   0.6967   0.3906   0.3906   0.5023   0.5023
negative_control               230   0.7667   0.7667   0.4374   0.4374   0.5852   0.5852
names_only                     167   0.5567   0.5567   0.3904   0.3904   0.2727   0.2727

All six conditions match the scoring record to four decimal places.
Baseline confusion matrix matches Experiment 01.


## 3. Predictions

작성 전: 실행 전에 지원이 직접 적는다

## 4. Paired Label Transitions

For each replacement condition, the code builds a 7 × 7 transition table from the paired labels.

- Rows = baseline label of the utterance.
- Columns = label of the same utterance under the condition.
- Row and column order is the tag order (NC, KET, GSR, RST, REV, PFA, PFR). Empty categories are shown as 0.

The diagonal counts utterances whose label did not change. Everything off the diagonal is a change.

The human label is not used in this section. A transition table shows where labels moved, not whether the move was right or wrong.

### 4.1. Replacement Conditions

Four conditions are compared with baseline: `definition_replacement`, `example_replacement`, `exclusion_rule_replacement`, and `negative_control`.

In [3]:
# 4.1. Transition tables — replacement conditions

def transition_table(df, cond):
    # Rows = baseline label, columns = condition label, fixed tag order with zeros for empty categories
    table = pd.crosstab(df["baseline"], df[cond])
    table = table.reindex(index=CATEGORIES, columns=CATEGORIES, fill_value=0)
    table.index.name = "baseline"
    table.columns.name = cond
    return table

transitions = {}
for cond in REPLACEMENTS:
    table = transition_table(labels, cond)
    assert table.values.sum() == 300
    unchanged = int(np.trace(table.values))
    changed = 300 - unchanged
    transitions[cond] = table

    print(f"baseline -> {cond}")
    print(table.to_string())
    print(f"Unchanged: {unchanged}    Changed: {changed}")
    print()

baseline -> definition_replacement
definition_replacement   NC  KET  GSR  RST  REV  PFA  PFR
baseline                                                 
NC                      172    2    1    1    2    1    0
KET                       4   42    0    0    0    0    0
GSR                       0    0    2    0    0    0    0
RST                       0    0    0    4    0    0    0
REV                       0    0    0    2   11    0    0
PFA                       0    0    0    0    0   46    0
PFR                       0    1    0    0    0    2    7
Unchanged: 284    Changed: 16

baseline -> example_replacement
example_replacement   NC  KET  GSR  RST  REV  PFA  PFR
baseline                                              
NC                   168    5    0    1    4    1    0
KET                    0   41    0    0    1    4    0
GSR                    0    1    1    0    0    0    0
RST                    0    0    0    3    1    0    0
REV                    0    0    0    0   13    0 

### 4.2. Names-Only Condition (Reference)

The names-only condition removes the manual entirely, so it is not a replacement of one manual part. Its transition table is shown here for reference only and is excluded from the comparisons in Section 5.

In [4]:
# 4.2. Transition table — names_only (reference only)

table_names_only = transition_table(labels, "names_only")
assert table_names_only.values.sum() == 300

print("baseline -> names_only")
print(table_names_only.to_string())
print(f"Unchanged: {int(np.trace(table_names_only.values))}    Changed: {300 - int(np.trace(table_names_only.values))}")

baseline -> names_only
names_only   NC  KET  GSR  RST  REV  PFA  PFR
baseline                                     
NC          127   25    5   13    0    9    0
KET          11   31    0    1    0    3    0
GSR           0    0    2    0    0    0    0
RST           0    0    0    4    0    0    0
REV           1    0    0    9    3    0    0
PFA          10    8    0    3    0   19    6
PFR           1    1    0    0    0    0    8
Unchanged: 194    Changed: 106


## 5. Classification Errors

This section brings the human label back in. For each utterance and each replacement condition, the code checks whether the baseline label and the condition label agree with the human label, and sorts the utterance into one of four classes.

| Class | Label changed? | Baseline | Condition |
|---|---|---|---|
| unchanged | No | — | — |
| wrong → right | Yes | ≠ human | = human |
| right → wrong | Yes | = human | ≠ human |
| wrong → wrong | Yes | ≠ human | ≠ human |

A changed label cannot go from right to right, because two labels that both equal the human label are the same label. The four classes therefore cover all 300 utterances.

The net change in agreement is (wrong → right) − (right → wrong). It must equal the condition's exact agreement count minus the baseline count of 236 from the scoring record.

### 5.1. Error Classification

In [5]:
# 5.1. Error classification

CLASSES = ["unchanged", "wrong→right", "right→wrong", "wrong→wrong"]

def classify(df, cond):
    changed = df["baseline"] != df[cond]
    base_right = df["baseline"] == df["human"]
    cond_right = df[cond] == df["human"]
    return pd.Series(np.select(
        [~changed, changed & ~base_right & cond_right, changed & base_right & ~cond_right],
        ["unchanged", "wrong→right", "right→wrong"],
        default="wrong→wrong",
    ), index=df.index, name="class")

error_classes = {}
rows = []
for cond in REPLACEMENTS:
    cls = classify(labels, cond)
    counts = cls.value_counts().reindex(CLASSES, fill_value=0)
    net = int(counts["wrong→right"] - counts["right→wrong"])

    # Verify against Section 4 and the scoring record
    assert counts.sum() == 300
    assert counts["unchanged"] == np.trace(transitions[cond].values)
    assert counts["right→wrong"] - counts["wrong→right"] == record.loc["baseline", "agree"] - record.loc[cond, "agree"], cond

    error_classes[cond] = cls
    rows.append({"condition": cond, **counts.to_dict(), "net change in agreement": net})

comparison = pd.DataFrame(rows).set_index("condition")

print(comparison.to_string())
print()
print("Net change in agreement equals the scoring-record agreement count minus 236 for all four conditions.")

                            unchanged  wrong→right  right→wrong  wrong→wrong  net change in agreement
condition                                                                                            
definition_replacement            284            6            7            3                       -1
example_replacement               276            3           16            5                      -13
exclusion_rule_replacement        261            4           31            4                      -27
negative_control                  284            3            9            4                       -6

Net change in agreement equals the scoring-record agreement count minus 236 for all four conditions.


### 5.2. Most Frequent Transitions

For each condition, the five most frequent changes (from baseline label to condition label) are listed with their counts.

One transition cell can contain utterances with different human labels, so the same from → to pair can be right → wrong for some utterances and wrong → right or wrong → wrong for others. The three columns split the count by class.

In [6]:
# 5.2. Most frequent transitions per condition

CHANGE_CLASSES = CLASSES[1:]

for cond in REPLACEMENTS:
    changed = labels.loc[error_classes[cond] != "unchanged", ["baseline", cond]].copy()
    changed["class"] = error_classes[cond][changed.index]
    changed.columns = ["from", "to", "class"]

    top = (changed.groupby(["from", "to", "class"], observed=True).size()
           .unstack("class", fill_value=0)
           .reindex(columns=CHANGE_CLASSES, fill_value=0))
    top["count"] = top.sum(axis=1)
    top = top.reset_index()
    top["from_order"] = top["from"].map(CATEGORIES.index)
    top["to_order"] = top["to"].map(CATEGORIES.index)
    top = (top.sort_values(["count", "from_order", "to_order"], ascending=[False, True, True])
           .head(5)[["from", "to", "count", *CHANGE_CLASSES]])

    assert top["count"].sum() <= (error_classes[cond] != "unchanged").sum()

    print(f"{cond}: top 5 transitions (baseline -> {cond})")
    print(top.to_string(index=False))
    print()

definition_replacement: top 5 transitions (baseline -> definition_replacement)
from  to  count  wrong→right  right→wrong  wrong→wrong
 KET  NC      4            3            1            0
  NC KET      2            1            1            0
  NC REV      2            0            2            0
 REV RST      2            0            0            2
 PFR PFA      2            1            0            1

example_replacement: top 5 transitions (baseline -> example_replacement)
from  to  count  wrong→right  right→wrong  wrong→wrong
  NC KET      5            0            5            0
  NC REV      4            0            3            1
 KET PFA      4            0            4            0
 PFA PFR      4            0            2            2
  NC RST      1            0            1            0

exclusion_rule_replacement: top 5 transitions (baseline -> exclusion_rule_replacement)
from  to  count  wrong→right  right→wrong  wrong→wrong
  NC KET     21            2           19   

## 6. Observations

작성 전: 실행 결과를 보고 지원이 직접 적는다